# dim_channel

In [1]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

GOLD_PATH = Path("../data/gold/dimensions")
GOLD_PATH.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Generate Date Range
# ---------------------------------------------------------------------

start_date = "2022-01-01"
end_date = "2025-12-31"

date_df = pd.DataFrame({
    "date": pd.date_range(start=start_date, end=end_date)
})

# ---------------------------------------------------------------------
# Date Key
# ---------------------------------------------------------------------

date_df["date_sk"] = date_df["date"].dt.strftime("%Y%m%d").astype(int)

# ---------------------------------------------------------------------
# Date Attributes
# ---------------------------------------------------------------------

date_df["day"] = date_df["date"].dt.day

date_df["month"] = date_df["date"].dt.month

date_df["month_name"] = date_df["date"].dt.month_name()

date_df["quarter"] = "Q" + date_df["date"].dt.quarter.astype(str)

date_df["year"] = date_df["date"].dt.year

date_df["week_of_year"] = date_df["date"].dt.isocalendar().week.astype(int)

date_df["day_of_week"] = date_df["date"].dt.dayofweek + 1

date_df["day_name"] = date_df["date"].dt.day_name()

date_df["is_weekend"] = date_df["day_name"].isin(
    ["Saturday", "Sunday"]
)

# ---------------------------------------------------------------------
# Financial Year (India)
# ---------------------------------------------------------------------

def get_financial_year(date):
    if date.month >= 4:
        return f"FY{date.year}-{str(date.year + 1)[-2:]}"
    return f"FY{date.year - 1}-{str(date.year)[-2:]}"

date_df["financial_year"] = date_df["date"].apply(get_financial_year)

# ---------------------------------------------------------------------
# Reorder Columns
# ---------------------------------------------------------------------

date_df = date_df[
    [
        "date_sk",
        "date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year",
        "week_of_year",
        "day_of_week",
        "day_name",
        "is_weekend",
        "financial_year"
    ]
]

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------

date_df.to_csv(
    GOLD_PATH / "dim_date.csv",
    index=False
)

date_df.to_parquet(
    GOLD_PATH / "dim_date.parquet",
    index=False
)

print("✅ dim_date created successfully")
print(date_df.head())
print(f"\nTotal Records : {len(date_df):,}")

✅ dim_date created successfully
    date_sk       date  day  month month_name quarter  year  week_of_year  \
0  20220101 2022-01-01    1      1    January      Q1  2022            52   
1  20220102 2022-01-02    2      1    January      Q1  2022            52   
2  20220103 2022-01-03    3      1    January      Q1  2022             1   
3  20220104 2022-01-04    4      1    January      Q1  2022             1   
4  20220105 2022-01-05    5      1    January      Q1  2022             1   

   day_of_week   day_name  is_weekend financial_year  
0            6   Saturday        True      FY2021-22  
1            7     Sunday        True      FY2021-22  
2            1     Monday       False      FY2021-22  
3            2    Tuesday       False      FY2021-22  
4            3  Wednesday       False      FY2021-22  

Total Records : 1,461


# dim_channel

In [2]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold/dimensions")

GOLD_PATH.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Load Silver Table
# ------------------------------------------------------------------

channel = pd.read_csv(SILVER_PATH / "channel_lookup.csv")

# ------------------------------------------------------------------
# Create Surrogate Key
# ------------------------------------------------------------------

channel.insert(0, "channel_sk", range(1, len(channel) + 1))

# ------------------------------------------------------------------
# Channel Type
# ------------------------------------------------------------------

online_channels = [
    "Website",
    "Mobile App",
    "Aggregator"
]

channel["channel_type"] = channel["channel_name"].apply(
    lambda x: "Online" if x in online_channels else "Offline"
)

# ------------------------------------------------------------------
# Ownership
# ------------------------------------------------------------------

direct_channels = [
    "Website",
    "Mobile App",
    "Branch",
    "Call Center"
]

channel["ownership"] = channel["channel_name"].apply(
    lambda x: "Direct" if x in direct_channels else "Indirect"
)

# ------------------------------------------------------------------
# Digital Flag
# ------------------------------------------------------------------

digital_channels = [
    "Website",
    "Mobile App"
]

channel["is_digital"] = channel["channel_name"].isin(digital_channels)

# ------------------------------------------------------------------
# Priority
# ------------------------------------------------------------------

priority = {
    "Website": "High",
    "Mobile App": "High",
    "Agent": "High",
    "Aggregator": "Medium",
    "Partner": "Medium",
    "Branch": "Low",
    "Call Center": "Low"
}

channel["priority"] = channel["channel_name"].map(priority)

# ------------------------------------------------------------------
# Reorder Columns
# ------------------------------------------------------------------

channel = channel[
    [
        "channel_sk",
        "channel_id",
        "channel_name",
        "channel_type",
        "ownership",
        "priority",
        "is_digital"
    ]
]

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------

channel.to_csv(
    GOLD_PATH / "dim_channel.csv",
    index=False
)

channel.to_parquet(
    GOLD_PATH / "dim_channel.parquet",
    index=False
)

print("✅ dim_channel created successfully")
print(channel)

✅ dim_channel created successfully
     channel_sk  channel_id   channel_name channel_type ownership priority  \
0             1         1.0    Channel_1.0      Offline  Indirect      NaN   
1             2         2.0    Channel_2.0      Offline  Indirect      NaN   
2             3         3.0    Channel_3.0      Offline  Indirect      NaN   
3             4         4.0    Channel_4.0      Offline  Indirect      NaN   
4             5         6.0    Channel_6.0      Offline  Indirect      NaN   
..          ...         ...            ...          ...       ...      ...   
150         151       157.0  Channel_157.0      Offline  Indirect      NaN   
151         152       158.0  Channel_158.0      Offline  Indirect      NaN   
152         153       159.0  Channel_159.0      Offline  Indirect      NaN   
153         154       160.0  Channel_160.0      Offline  Indirect      NaN   
154         155       163.0  Channel_163.0      Offline  Indirect      NaN   

     is_digital  
0         

# dim_customer

['policy_id', 'airbags', 'is_esc', 'is_tpms', 'is_parking_sensors', 'is_parking_camera', 'is_brake_assist', 'is_power_steering', 'is_speed_alert', 'ncap_rating', 'safety_score']
